
# 📚 Card Cutter (DOCX → JSON) — **GPU + Parallel** Colab Notebook

This version:
- Parses `.docx` files **in parallel** using `ProcessPoolExecutor` (fast on 4000+ files).
- Uses your **A100 GPU** for **semantic deduplication** via `sentence-transformers` + cosine similarity.
- Falls back gracefully if FAISS-GPU isn't available (uses Torch on CUDA).


In [ ]:

# --- Setup ---
!pip -q install python-docx tqdm sentence-transformers
# Try FAISS for fast ANN search; fall back to CPU if GPU wheel isn't available
try:
    import faiss  # noqa
except Exception:
    try:
        !pip -q install faiss-gpu
    except Exception:
        !pip -q install faiss-cpu


In [ ]:

# --- Configuration ---
from pathlib import Path

# Input folder with your .docx files (change if needed)
DOCS_DIR = Path('/content/docs')

# Outputs
COMPILED_JSON = Path('/content/compiled_cards.json')
FINAL_JSON    = Path('/content/final_cards.json')
CLEANED_JSON  = Path('/content/cleaned_cards.json')

# Embedding model & dedupe threshold (cosine)
EMBED_MODEL   = 'sentence-transformers/all-mpnet-base-v2'
SIM_THRESHOLD = 0.92

# Parallel workers (None -> use all CPU cores)
MAX_WORKERS = None

DOCS_DIR.mkdir(parents=True, exist_ok=True)
print('Docs dir:', DOCS_DIR.absolute())
print('Outputs :', COMPILED_JSON, FINAL_JSON, CLEANED_JSON, sep='\n')

# OPTIONAL: Mount Google Drive if your docs live there
# from google.colab import drive
# drive.mount('/content/drive')
# DOCS_DIR = Path('/content/drive/MyDrive/path/to/docs')


In [ ]:

# --- Core types & helpers ---
from dataclasses import dataclass, asdict
from typing import List, Tuple

@dataclass
class Card:
    tag: str
    cite: str
    text_body: str
    underline_ranges: List[Tuple[int, int]]
    emphasis_ranges: List[Tuple[int, int]]
    highlight_ranges: List[Tuple[int, int]]

    def to_dict(self):
        return asdict(self)

def _merge_ranges(ranges: List[Tuple[int,int]]) -> List[Tuple[int,int]]:
    if not ranges: return []
    ranges = sorted(ranges)
    merged = [ranges[0]]
    for s,e in ranges[1:]:
        ls, le = merged[-1]
        if s <= le:
            merged[-1] = (ls, max(le, e))
        else:
            merged.append((s,e))
    return merged

def _append_span(store: List[Tuple[int,int]], s: int, e: int):
    if s < e:
        store.append((s,e))


In [ ]:

# --- DOCX Parser (python-docx) ---
import docx

MARKER_STYLES = {'Tag','Pocket','Hat','Block','Heading'}

def _style_contains(style, needle: str) -> bool:
    try:
        return (style.name or '').lower().__contains__(needle.lower())
    except Exception:
        return False

def _iter_runs_with_indices(paragraph, base_offset: int):
    off = base_offset
    for run in paragraph.runs:
        t = run.text or ''
        start, end = off, off + len(t)
        yield t, start, end, run
        off = end

def _collect_body_and_ranges(paragraphs, start_idx: int):
    body_parts, underline, emphasis, highlight = [], [], [], []
    offset, i = 0, start_idx
    while i < len(paragraphs):
        p = paragraphs[i]
        sname = getattr(p.style, 'name', '') or ''
        if any(m.lower() in sname.lower() for m in MARKER_STYLES):
            break
        ptext = p.text or ''
        for t, s, e, run in _iter_runs_with_indices(p, offset):
            if not t: continue
            try:
                if bool(run.underline): _append_span(underline, s, e)
            except: pass
            try:
                if bool(run.italic) or _style_contains(run.style, 'emphasis'):
                    _append_span(emphasis, s, e)
            except: pass
            try:
                if getattr(run.font, 'highlight_color', None) is not None:
                    _append_span(highlight, s, e)
            except: pass
        body_parts.append(ptext)
        offset += len(ptext)
        if i < len(paragraphs) - 1:
            body_parts.append('\n'); offset += 1
        i += 1
    text = ''.join(body_parts).strip()
    return text, _merge_ranges(underline), _merge_ranges(emphasis), _merge_ranges(highlight), i

def extract_cards_from_word_doc(doc_path: str):
    doc = docx.Document(doc_path)
    paragraphs = doc.paragraphs
    cards, i, N = [], 0, len(paragraphs)
    while i < N:
        p = paragraphs[i]
        sname = getattr(p.style, 'name', '') or ''
        if 'tag' in sname.lower():
            tag = (p.text or '').strip()
            if i+1 < N:
                cite_p = paragraphs[i+1]
                cite_style = getattr(cite_p.style, 'name', '') or ''
                if ('cite' in cite_style.lower()) or ('normal/card' in cite_style.lower()) or ('quote' in cite_style.lower()):
                    cite = (cite_p.text or '').strip()
                    body, u, em, hi, next_i = _collect_body_and_ranges(paragraphs, i+2)
                    if tag and cite and body:
                        cards.append(Card(tag=tag, cite=cite, text_body=body,
                                          underline_ranges=u, emphasis_ranges=em, highlight_ranges=hi))
                    i = max(next_i, i+2)
                else:
                    i += 1
            else:
                i += 1
        else:
            i += 1
    return cards


In [ ]:

# --- Parallel compile from folder ---
import os
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

def list_docx_files(folder: Path):
    return [str((folder/f).resolve()) for f in os.listdir(folder) if f.lower().endswith('.docx')]

def _process_one_file(path: str):
    try:
        res = extract_cards_from_word_doc(path)
        return [c.to_dict() if hasattr(c, 'to_dict') else c for c in res]
    except Exception as e:
        return {'__error__': str(e), 'file': path}

def compile_cards_parallel(folder: Path, max_workers=None):
    files = list_docx_files(folder)
    if not files:
        print(f'No .docx files in {folder}'); return []
    all_cards, errors = [], []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_process_one_file, fp): fp for fp in files}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Parsing DOCX (parallel)'):
            res = fut.result()
            if isinstance(res, list):
                all_cards.extend(res)
            else:
                errors.append(res)
    if errors:
        print(f'Finished with {len(errors)} file errors (showing up to 5):', errors[:5])
    return all_cards


In [ ]:

# --- GPU Embedding & Dedupe ---
import torch, numpy as np, json
from sentence_transformers import SentenceTransformer

try:
    import faiss
    _faiss_ok = True
except Exception:
    _faiss_ok = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch device:', device)

model = SentenceTransformer(EMBED_MODEL, device=device)
print('Loaded model:', EMBED_MODEL)

def embed_texts(texts, batch_size=256):
    embs = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        device=device
    )
    return embs.astype('float32')

def gpu_deduplicate(cards, embs, sim_thresh=0.92):
    order = np.argsort([-len(c.get('text_body','')) for c in cards])
    kept, kept_texts, kept_embs = [], [], []

    use_faiss = False
    if _faiss_ok:
        try:
            d = embs.shape[1]
            index = faiss.IndexFlatIP(d)  # cosine on normalized vectors
            if hasattr(faiss, 'get_num_gpus') and faiss.get_num_gpus() > 0:
                index = faiss.index_cpu_to_all_gpus(index)
            use_faiss = True
        except Exception as e:
            print('FAISS not on GPU; using Torch. Reason:', e)

    for idx in order:
        txt = (cards[idx].get('text_body') or '')
        if not txt:
            continue

        # quick substring filter
        if any((txt in t) or (t in txt) for t in kept_texts):
            continue

        if kept_embs:
            vec = embs[idx].reshape(1, -1)
            if use_faiss:
                D, I = index.search(vec, 1)
                if D[0][0] >= sim_thresh:
                    continue
            else:
                K = torch.from_numpy(np.vstack(kept_embs)).to(device)
                v = torch.from_numpy(vec).to(device)
                sims = torch.matmul(K, v.T).squeeze(1)
                if float(torch.max(sims).item()) >= sim_thresh:
                    continue

        kept.append(cards[idx])
        kept_texts.append(txt)
        kept_embs.append(embs[idx])
        if use_faiss:
            index.add(embs[idx].reshape(1, -1))

    return kept


In [ ]:

# --- Run the pipeline ---
import json

# 1) Compile in parallel
all_cards = compile_cards_parallel(DOCS_DIR, max_workers=MAX_WORKERS)
print('Compiled cards:', len(all_cards))
with open(COMPILED_JSON, 'w', encoding='utf-8') as f:
    json.dump(all_cards, f, indent=2, ensure_ascii=False)
print('Wrote', COMPILED_JSON)

# 2) GPU deduplication
texts = [c.get('text_body','') for c in all_cards]
embs  = embed_texts(texts, batch_size=256)
deduped = gpu_deduplicate(all_cards, embs, sim_thresh=SIM_THRESHOLD)
print('After GPU deduplication:', len(deduped))
with open(FINAL_JSON, 'w', encoding='utf-8') as f:
    json.dump(deduped, f, indent=2, ensure_ascii=False)
print('Wrote', FINAL_JSON)

# 3) Clean
REQUIRED_FIELDS = ["tag","cite","text_body","underline_ranges","emphasis_ranges","highlight_ranges"]
def is_card_complete(c):
    for k in REQUIRED_FIELDS:
        v = c.get(k)
        if not v: return False
        if isinstance(v, str) and not v.strip(): return False
    return True
cleaned = [c for c in deduped if is_card_complete(c)]
print('After cleaning:', len(cleaned))
with open(CLEANED_JSON, 'w', encoding='utf-8') as f:
    json.dump(cleaned, f, indent=2, ensure_ascii=False)
print('Wrote', CLEANED_JSON)

if cleaned:
    print('\nSample card:')
    print(json.dumps(cleaned[0], indent=2, ensure_ascii=False)[:1500])
else:
    print('No cards parsed.')
